In [1]:
#Step 2 Import Libraries
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.documents import Document

from langchain_core.prompts import PromptTemplate

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_classic.chains.retrieval import create_retrieval_chain

from langchain_community.llms import Ollama

from langchain_ollama import ChatOllama

import chromadb

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/61/1rk9vp2x42540cvytqtfywdm0000gn/T/ipykernel_35064/2801644876.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama


In [3]:
#Step 2 Load Dataset
DATA_PATH = "../data/processed/cleaned_patent_data.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(18352, 30)

In [4]:
df.columns

Index(['source_url', 'patent_id', 'title', 'title_en', 'abstract',
       'abstract_en', 'description', 'claims', 'filing_date',
       'publication_date', 'country', 'assignee_original', 'assignee_en',
       'ipc_codes', 'cpc_codes', 'claim_count', 'independent_claim_count',
       'backward_citation_count', 'forward_citation_count', 'legal_status',
       'detected_language', 'claims_clean', 'combined_text', 'abstract_length',
       'word_count', 'claims_length_chars', 'claims_word_count',
       'claims_length', 'filing_year', 'year'],
      dtype='object')

In [ ]:
#Step 4 Load Saved Embeddings (here we are using embeddings that were created during traditional rag)
embeddings = np.load(
    "../data/processed/chunk_embeddings.npy"
)

embeddings.shape

NameError: name 'np' is not defined

In [7]:
#Step 5 Convert Dataset into LangChain Documents
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    content = f"""
Title: {row['title']}

Abstract:
{row['abstract_en']}

Claims:
{row['claims_clean']}

Country:
{row['country']}

Assignee:
{row['assignee_en']}

Filing Year:
{row['filing_year']}

Forward Citations:
{row['forward_citation_count']}

Backward Citations:
{row['backward_citation_count']}


"""

    documents.append(
        Document(
            page_content=content,
            metadata={
                "patent_id": row["patent_id"],
                "title": row["title"],
                "country": row["country"],
                "filing_year": row["filing_year"],
                
            },
        )
    )

print(len(documents))

18352


In [8]:
#Step 6 Load Embedding Model
embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4389.89it/s]


In [9]:
#Step 7 Create Chroma Vector Database
vector_db = Chroma.from_documents(

    documents=documents,

    embedding=embedding_model,

    persist_directory="../vector_db/chroma_db"
)

In [10]:
print(vector_db._collection.count())

18352


In [11]:
#Step 8 Create Retriever
retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={

        "k":5

    }

)

In [12]:
#Step 9 Test Retrieval
query = "AI wearable glucose monitoring"

docs = retriever.invoke(query)

for doc in docs:

    print(doc.metadata)

    print(doc.page_content[:300])

    print("="*80)

{'filing_year': 2016.0, 'patent_id': 'KR101879940B1', 'title': 'Pilot Bio-Signal Monitoring System using Wearable Continuous Body Fluid checking apparatus', 'country': 'KR'}

Title: Pilot Bio-Signal Monitoring System using Wearable Continuous Body Fluid checking apparatus

Abstract:
The present invention relates to a system for detecting a bio-signal of a pilot using a wearable continuous body fluid check system, which comprises a body fluid sensor 110 configured to be 
{'country': 'US', 'filing_year': 2019.0, 'patent_id': 'US20210068723A1', 'title': 'Intelligent prediction-based glucose alarm devices, systems, and methods'}

Title: Intelligent prediction-based glucose alarm devices, systems, and methods

Abstract:
Glucose-related alarms may be generated based on glucose predictions, which may be short-to-medium term. The glucose predictions, in turn, may be computed based on a multiplicity of input data sources which m
{'country': 'CN', 'patent_id': 'CN114141331A', 'filing_year': 2021

In [13]:
#Step 10 Load Local LLM
llm = ChatOllama(
    model="llama3:latest",
    temperature=0)

In [14]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}} output_version=None model='llama3:latest' temperature=0.0


In [15]:
response = llm.invoke("Hello")

print(response.content)

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [16]:
# Step 11 Prompt Template
template = """
You are a Medical Patent Search Assistant.

Answer ONLY using the retrieved patent documents.

Rules:

- Never invent information.
- Never explain unless asked.
- Never summarize unless asked.
- Return exactly what the user requests.

Examples:

User:
List AI diagnostic medical patents

Assistant:
Patent A
Patent B
Patent C

User:
Who is the assignee?

Assistant:
Siemens Healthineers

User:
Which patents were filed in 2023?

Assistant:
Patent X
Patent Y

Context:
{context}

Question:
{input}

Answer:
"""

In [17]:
#Create template
prompt = PromptTemplate(

    template=template,

    input_variables=[

        "context",

        "input"

    ]
)

In [18]:
# Step 12 Build RAG Retrieval Chain

question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt
)

qa_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [19]:
query = "What are the latest AI diagnostic medical device patents?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):

    print("DOCUMENT", i+1)
    print(doc.metadata)
    print(doc.page_content[:1000])
    print("="*100)

DOCUMENT 1
{'patent_id': 'JP7368049B2', 'filing_year': 2020.0, 'country': 'JP', 'title': 'Health management device, health management method, health management system, program, and recording medium'}

Title: Health management device, health management method, health management system, program, and recording medium

Abstract:
nan

Claims:
Contains foreign object information, In addition, it includes a recommended behavior guide section, The recommended action guide section is Generating recommended behavior information based on the excretion information based on the integrated biological information integrated by the information integration unit, Generating a pattern of excretion time of the health management subject as incontinence improvement behavior information based on the integrated biological information, The recommended behavior information based on the excretion information includes the incontinence improvement behavior information, The information output unit outputs the recom

In [21]:
# Step 13 Ask Questions

response = qa_chain.invoke(
    {
        "input": "What are recent wearable biosensor patents??"
    }
)

print(response["answer"])

Based on the retrieved patent documents, here are some recent wearable biosensor patents:

1. **WO 2016/000123**: A biosensor device and physiological monitor that includes a flexible base with a first sensor body and a second sensor body detachably mounted at intervals on the first side of the flexible base.
2. **US 10,345,351**: A physiological monitor that includes a biosensor device with a flexible base, a first sensor body, and a second sensor body, as well as a signal processing device and a monitor.

These patents describe wearable biosensors that can measure different physiological parameters, such as tissue blood oxygen levels and anesthetic depth. The sensors are detachably mounted on a flexible base, allowing for easy use and reconfiguration.


In [22]:
# Step 14 Display Retrieved Sources

for doc in response["context"]:

    print(doc.metadata)

    print("="*80)

{'country': 'WO', 'filing_year': 2020.0, 'title': 'Smart-clothes-coupling-type biosignal measurement system', 'patent_id': 'WO2022114293A1'}
{'patent_id': 'KR20170116458A', 'title': 'Wearable biodevice and manufacturing method thereof', 'filing_year': 2016.0, 'country': 'KR'}
{'filing_year': 2019.0, 'patent_id': 'US20210361164A1', 'country': 'US', 'title': 'Medical biosensor device, system, and method'}
{'title': 'Biosensor and manufacturing method therefor', 'patent_id': 'WO2018012692A1', 'filing_year': 2016.0, 'country': 'WO'}
{'country': 'US', 'title': 'Biosensor device and physiological monitor', 'filing_year': 2019.0, 'patent_id': 'US10517492B2'}
